In [ ]:
!rm -rf /kaggle/working/prep2d /kaggle/working/prep2d_*

In [ ]:
import os, ast, glob
import numpy as np
import pandas as pd
import pydicom
import cv2

INPUT = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
OUTPUT = "/kaggle/working/prep2d"
SIZE = 512          
SIGMA_MM = 2.0
FIX_ONLY = True     


def from_sequence(item, sequence, attribute):
    if item is not None and sequence in item:
        return item[sequence][0].get(attribute)
    return None


def read_frames(ds):
    shared = ds["SharedFunctionalGroupsSequence"][0] if "SharedFunctionalGroupsSequence" in ds else None
    orientation = ds.get("ImageOrientationPatient") or from_sequence(shared, "PlaneOrientationSequence", "ImageOrientationPatient")
    spacing = ds.get("PixelSpacing") or from_sequence(shared, "PixelMeasuresSequence", "PixelSpacing")
    slope = ds.get("RescaleSlope") or from_sequence(shared, "PixelValueTransformationSequence", "RescaleSlope") or 1
    intercept = ds.get("RescaleIntercept") or from_sequence(shared, "PixelValueTransformationSequence", "RescaleIntercept") or 0
    pixels = ds.pixel_array
    if pixels.ndim == 2:
        pixels, positions = pixels[None], [ds.get("ImagePositionPatient")]
    else:
        per_frame = ds.get("PerFrameFunctionalGroupsSequence", [])
        positions = [from_sequence(f, "PlanePositionSequence", "ImagePositionPatient") for f in per_frame]
        positions = positions if len(positions) == len(pixels) else [None] * len(pixels)
    return [{"key": (ds.SOPInstanceUID, f), "position": positions[f], "orientation": orientation, "spacing": spacing,
             "instance": int(ds.get("InstanceNumber", 0) or 0),
             "pixels": pixels[f] * float(slope) + float(intercept)} for f in range(len(pixels))]


def load_series(uid):
    files = [pydicom.dcmread(f) for f in glob.glob(f"{INPUT}/series/{uid}/*.dcm")]
    frames = [frame for ds in files for frame in read_frames(ds)]
    orientation = frames[0]["orientation"] or [1, 0, 0, 0, 1, 0]
    normal = np.cross(np.array(orientation[:3], float), np.array(orientation[3:], float))
    if all(f["position"] is not None for f in frames):
        for f in frames:
            f["z"] = float(np.dot(np.array(f["position"], float), normal))
    else:
        spacing = float(files[0].get("SpacingBetweenSlices") or files[0].get("SliceThickness") or 1)
        for i, f in enumerate(sorted(frames, key=lambda f: (f["instance"], f["key"][1]))):
            f["z"] = i * spacing
    frames.sort(key=lambda f: f["z"])
    slices = np.stack([f["pixels"] for f in frames]).astype(np.float32)
    keys = [f["key"] for f in frames]
    z = np.array([f["z"] for f in frames])
    patient = str(files[0].get("PatientID", ""))
    spacing = [float(v) for v in frames[0]["spacing"]] if frames[0]["spacing"] else [1.0, 1.0]
    return slices, keys, z, spacing, patient


def normalize(slices, modality):
    if modality == "CTA":
        return np.clip(slices, 0, 900) / 900 * 2 - 1
    nonzero = slices[slices > 0]
    return np.where(slices > 0, (slices - nonzero.mean()) / nonzero.std(), 0)


def pad_or_crop(image, background):
    out = np.full((SIZE, SIZE), background, np.float32)
    h, w = image.shape
    src_top, src_left = max(0, (h - SIZE) // 2), max(0, (w - SIZE) // 2)
    dst_top, dst_left = max(0, (SIZE - h) // 2), max(0, (SIZE - w) // 2)
    ch, cw = min(h, SIZE), min(w, SIZE)
    out[dst_top:dst_top + ch, dst_left:dst_left + cw] = image[src_top:src_top + ch, src_left:src_left + cw]
    return out


def fit(slices):
    h, w = slices.shape[1:]
    scale = min(1.0, SIZE / max(h, w))
    background = float(slices.min())
    out = []
    for s in slices:
        if scale < 1:
            s = cv2.resize(s.astype(np.float32), (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
        out.append(pad_or_crop(s, background))
    return np.stack(out).astype(np.float16), scale


def make_labels(keys, z, aneurysms):
    labels = np.zeros(len(keys), np.float32)
    for key in aneurysms:
        if key in keys:
            distance = z - z[keys.index(key)]
            labels = np.maximum(labels, np.exp(-distance ** 2 / (2 * SIGMA_MM ** 2)))
    labels[labels < 0.05] = 0
    return labels.round(3)


def hu_shift(slices, modality):
    if modality == "CTA" and np.percentile(slices[:, ::4, ::4], 90) < -500:
        return 1024
    return 0


def process_series(uid, modality, aneurysms):
    slices, keys, z, spacing, patient = load_series(uid)
    shift = hu_shift(slices, modality)
    slices = slices + shift
    original_shape = slices.shape[1:]
    slices, scale = fit(normalize(slices, modality))
    labels = make_labels(keys, z, aneurysms)
    np.savez_compressed(f"{OUTPUT}/{uid}.npz", slices=slices, labels=labels,
                        original_spacing=spacing, original_shape=original_shape, scale=scale, hu_shift=shift)
    return [{"uid": uid, "patient": patient, "modality": modality, "slice": k, "label": float(labels[k]),
             "spacing": spacing[0] / scale, "hu_shift": shift} for k in range(len(keys))]


def add_split(slices_csv, split_csv):
    slices = pd.read_csv(slices_csv)
    split = pd.read_csv(split_csv)[["uid", "split"]]
    return slices.merge(split, on="uid")


def main(start, end):
    os.makedirs(OUTPUT, exist_ok=True)
    train = pd.read_csv(f"{INPUT}/train.csv")
    train = train[train["Modality"].isin(["CTA", "MRA"])].sort_values("SeriesInstanceUID")
    if FIX_ONLY:
        fix = pd.read_csv(glob.glob("/kaggle/input/**/hu_fix_uids.csv", recursive=True)[0])
        train = train[train["SeriesInstanceUID"].isin(set(fix["uid"]))]
    localizers = pd.read_csv(f"{INPUT}/train_localizers.csv")

    rows = []
    for uid, modality in zip(train["SeriesInstanceUID"][start:end], train["Modality"][start:end]):
        try:
            points = localizers[localizers["SeriesInstanceUID"] == uid]
            aneurysms = [(sop, int(ast.literal_eval(c).get("f", 0))) for sop, c in zip(points["SOPInstanceUID"], points["coordinates"])]
            series_rows = process_series(uid, modality, aneurysms)
            rows += series_rows
            print(uid, modality, len(series_rows), "felii,", sum(r["label"] >= 0.5 for r in series_rows), "cu eticheta >= 0.5,",
                  "hu_shift", series_rows[0]["hu_shift"] if series_rows else 0, flush=True)
        except Exception as e:
            print(uid, "EROARE", e, flush=True)

    pd.DataFrame(rows).to_csv(f"{OUTPUT}/slices_{start}_{end}.csv", index=False)
    print(f"gata: {len(rows)} felii salvate in slices_{start}_{end}.csv")

In [ ]:
import os, json, shutil, subprocess
from kaggle_secrets import UserSecretsClient

USERNAME = "tefanmateibliah"
PREFIX = "rsna-2d-fix"                                  
CHUNKS = [(i, i + 250) for i in range(0, 750, 250)]    

secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = USERNAME
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")


def save_dataset(folder, start, end):
    slug = f"{PREFIX}-{start}-{end}"
    metadata = {"title": f"RSNA 2D slices fix {start}-{end}", "id": f"{USERNAME}/{slug}", "licenses": [{"name": "unknown"}]}
    with open(f"{folder}/dataset-metadata.json", "w") as f:
        json.dump(metadata, f)
    result = subprocess.run(["kaggle", "datasets", "create", "-p", folder], capture_output=True, text=True)
    print(result.stdout[-500:], result.stderr[-500:], flush=True)
    return result.returncode == 0


for start, end in CHUNKS:
    OUTPUT = f"/kaggle/working/prep2d_{start}_{end}"
    main(start, end)
    ok = save_dataset(OUTPUT, start, end)
    print(f"[ DATASET ] {PREFIX}-{start}-{end} | {'creat' if ok else 'EROARE la upload, fisierele raman in ' + OUTPUT}", flush=True)
    if ok:
        shutil.rmtree(OUTPUT)